In [1]:
pip install polars

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pymc

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pytensor

Note: you may need to restart the kernel to use updated packages.


In [4]:
import arviz as az
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import seaborn.objects as so
import numpy as np
import pandas as pd
import polars as pl
import pymc as pm
import pytensor.tensor as pt
import random

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_SEED = 42

print(f"Running on PyMC v{pm.__version__}")


Running on PyMC v6.3.2


In [5]:
df = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/MNCAATourneyDetailedResults.csv")
df.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,134,1421,92,1411,84,N,1,32,69,...,31,14,31,17,28,16,15,5,0,22
1,2003,136,1112,80,1436,51,N,0,31,66,...,16,7,7,8,26,12,17,10,3,15
2,2003,136,1113,84,1272,71,N,0,31,59,...,28,14,21,20,22,11,12,2,5,18
3,2003,136,1141,79,1166,73,N,0,29,53,...,17,12,17,14,17,20,21,6,6,21
4,2003,136,1143,76,1301,74,N,1,27,64,...,21,15,20,10,26,16,14,5,8,19


In [6]:
df_city = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/Cities.csv")
df_city.head()

,CityID,City,State
0,4001,Abilene,TX
1,4002,Akron,OH
2,4003,Albany,NY
3,4004,Albuquerque,NM
4,4005,Allentown,PA


In [7]:
df_teamname = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/Mteams.csv")
df_teamname.head()

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026


In [8]:
df_gamecities = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/MGameCities.csv")
df_gamecities = df_gamecities[df_gamecities["CRType"] == "NCAA"]
df_gamecities

,Season,DayNum,WTeamID,LTeamID,CRType,CityID
5263,2010,134,1115,1457,NCAA,4091
5295,2010,136,1124,1358,NCAA,4237
5296,2010,136,1139,1431,NCAA,4305
5297,2010,136,1140,1196,NCAA,4254
5298,2010,136,1242,1250,NCAA,4254
...,...,...,...,...,...,...
86768,2025,146,1120,1277,NCAA,4016
86769,2025,146,1222,1397,NCAA,4161
86785,2025,152,1196,1120,NCAA,4302
86786,2025,152,1222,1181,NCAA,4302


In [9]:
teams = df_teamname[["TeamID", "TeamName"]]
teams


,TeamID,TeamName
0,1101,Abilene Chr
1,1102,Air Force
2,1103,Akron
3,1104,Alabama
4,1105,Alabama A&M
...,...,...
376,1477,East Texas A&M
377,1478,Le Moyne
378,1479,Mercyhurst
379,1480,West Georgia


In [10]:
df_seed = pd.read_csv("/Users/manasacharya/Desktop/data science capstone/march-machine-learning-mania-2026/MNCAATourneySeeds.csv")
df_seed

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
2689,2026,Z12,1219
2690,2026,Z13,1218
2691,2026,Z14,1244
2692,2026,Z15,1474


In [11]:
df = df.merge(teams.rename(columns={"TeamID": "WTeamID", "TeamName": "WTeamName"}), on="WTeamID", how="left")
df = df.merge(teams.rename(columns={"TeamID": "LTeamID", "TeamName": "LTeamName"}), on="LTeamID", how="left")

In [12]:
seeds = df_seed[["Season", "Seed", "TeamID"]]

df = df.merge(seeds.rename(columns={"TeamID": "WTeamID", "Seed": "WSeed"}), on=["Season", "WTeamID"], how="left")
df = df.merge(seeds.rename(columns={"TeamID": "LTeamID", "Seed": "LSeed"}), on=["Season", "LTeamID"], how="left")

df.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LDR,LAst,LTO,LStl,LBlk,LPF,WTeamName,LTeamName,WSeed,LSeed
0,2003,134,1421,92,1411,84,N,1,32,69,...,28,16,15,5,0,22,UNC Asheville,TX Southern,X16b,X16a
1,2003,136,1112,80,1436,51,N,0,31,66,...,26,12,17,10,3,15,Arizona,Vermont,Z01,Z16
2,2003,136,1113,84,1272,71,N,0,31,59,...,22,11,12,2,5,18,Arizona St,Memphis,Z10,Z07
3,2003,136,1141,79,1166,73,N,0,29,53,...,17,20,21,6,6,21,C Michigan,Creighton,Z11,Z06
4,2003,136,1143,76,1301,74,N,1,27,64,...,26,16,14,5,8,19,California,NC State,W08,W09


In [13]:
df = df.merge(
    df_gamecities[["Season", "DayNum", "WTeamID", "LTeamID", "CityID"]],
    on=["Season", "DayNum", "WTeamID", "LTeamID"],
    how="left",
)
df = df.merge(df_city, on="CityID", how="left")

df.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LStl,LBlk,LPF,WTeamName,LTeamName,WSeed,LSeed,CityID,City,State
0,2003,134,1421,92,1411,84,N,1,32,69,...,5,0,22,UNC Asheville,TX Southern,X16b,X16a,NaN,NaN,NaN
1,2003,136,1112,80,1436,51,N,0,31,66,...,10,3,15,Arizona,Vermont,Z01,Z16,NaN,NaN,NaN
2,2003,136,1113,84,1272,71,N,0,31,59,...,2,5,18,Arizona St,Memphis,Z10,Z07,NaN,NaN,NaN
3,2003,136,1141,79,1166,73,N,0,29,53,...,6,6,21,C Michigan,Creighton,Z11,Z06,NaN,NaN,NaN
4,2003,136,1143,76,1301,74,N,1,27,64,...,5,8,19,California,NC State,W08,W09,NaN,NaN,NaN


In [14]:
df

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LStl,LBlk,LPF,WTeamName,LTeamName,WSeed,LSeed,CityID,City,State
0,2003,134,1421,92,1411,84,N,1,32,69,...,5,0,22,UNC Asheville,TX Southern,X16b,X16a,NaN,NaN,NaN
1,2003,136,1112,80,1436,51,N,0,31,66,...,10,3,15,Arizona,Vermont,Z01,Z16,NaN,NaN,NaN
2,2003,136,1113,84,1272,71,N,0,31,59,...,2,5,18,Arizona St,Memphis,Z10,Z07,NaN,NaN,NaN
3,2003,136,1141,79,1166,73,N,0,29,53,...,6,6,21,C Michigan,Creighton,Z11,Z06,NaN,NaN,NaN
4,2003,136,1143,76,1301,74,N,1,27,64,...,5,8,19,California,NC State,W08,W09,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,2025,146,1120,70,1277,64,N,0,26,61,...,5,5,18,Auburn,Michigan St,Y01,Y02,4016.0,Atlanta,GA
1445,2025,146,1222,69,1397,50,N,0,28,66,...,3,2,9,Houston,Tennessee,X01,X02,4161.0,Indianapolis,IN
1446,2025,152,1196,79,1120,73,N,0,25,53,...,11,3,22,Florida,Auburn,Z01,Y01,4302.0,San Antonio,TX
1447,2025,152,1222,70,1181,67,N,0,23,61,...,7,4,16,Houston,Duke,X01,W01,4302.0,San Antonio,TX


In [15]:
df_final = df[["Season","WScore","LScore","WTeamName","LTeamName","WSeed","LSeed"]]
df_final

,Season,WScore,LScore,WTeamName,LTeamName,WSeed,LSeed
0,2003,92,84,UNC Asheville,TX Southern,X16b,X16a
1,2003,80,51,Arizona,Vermont,Z01,Z16
2,2003,84,71,Arizona St,Memphis,Z10,Z07
3,2003,79,73,C Michigan,Creighton,Z11,Z06
4,2003,76,74,California,NC State,W08,W09
...,...,...,...,...,...,...,...
1444,2025,70,64,Auburn,Michigan St,Y01,Y02
1445,2025,69,50,Houston,Tennessee,X01,X02
1446,2025,79,73,Florida,Auburn,Z01,Y01
1447,2025,70,67,Houston,Duke,X01,W01


In [16]:
df["WSeedNum"] = df["WSeed"].str[1:3].astype(int)
df["LSeedNum"] = df["LSeed"].str[1:3].astype(int)
df

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LPF,WTeamName,LTeamName,WSeed,LSeed,CityID,City,State,WSeedNum,LSeedNum
0,2003,134,1421,92,1411,84,N,1,32,69,...,22,UNC Asheville,TX Southern,X16b,X16a,NaN,NaN,NaN,16,16
1,2003,136,1112,80,1436,51,N,0,31,66,...,15,Arizona,Vermont,Z01,Z16,NaN,NaN,NaN,1,16
2,2003,136,1113,84,1272,71,N,0,31,59,...,18,Arizona St,Memphis,Z10,Z07,NaN,NaN,NaN,10,7
3,2003,136,1141,79,1166,73,N,0,29,53,...,21,C Michigan,Creighton,Z11,Z06,NaN,NaN,NaN,11,6
4,2003,136,1143,76,1301,74,N,1,27,64,...,19,California,NC State,W08,W09,NaN,NaN,NaN,8,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,2025,146,1120,70,1277,64,N,0,26,61,...,18,Auburn,Michigan St,Y01,Y02,4016.0,Atlanta,GA,1,2
1445,2025,146,1222,69,1397,50,N,0,28,66,...,9,Houston,Tennessee,X01,X02,4161.0,Indianapolis,IN,1,2
1446,2025,152,1196,79,1120,73,N,0,25,53,...,22,Florida,Auburn,Z01,Y01,4302.0,San Antonio,TX,1,1
1447,2025,152,1222,70,1181,67,N,0,23,61,...,16,Houston,Duke,X01,W01,4302.0,San Antonio,TX,1,1


In [17]:
df_final = df[["Season","WScore","LScore","WTeamName","LTeamName","WSeedNum","LSeedNum"]]
df_final

,Season,WScore,LScore,WTeamName,LTeamName,WSeedNum,LSeedNum
0,2003,92,84,UNC Asheville,TX Southern,16,16
1,2003,80,51,Arizona,Vermont,1,16
2,2003,84,71,Arizona St,Memphis,10,7
3,2003,79,73,C Michigan,Creighton,11,6
4,2003,76,74,California,NC State,8,9
...,...,...,...,...,...,...,...
1444,2025,70,64,Auburn,Michigan St,1,2
1445,2025,69,50,Houston,Tennessee,1,2
1446,2025,79,73,Florida,Auburn,1,1
1447,2025,70,67,Houston,Duke,1,1


In [18]:
wins = df[["Season", "WSeedNum", "LSeedNum"]].rename(columns={"WSeedNum": "team1", "LSeedNum": "team2"})
wins["outcome"] = 1

losses = df[["Season", "WSeedNum", "LSeedNum"]].rename(columns={"LSeedNum": "team1", "WSeedNum": "team2"})
losses["outcome"] = 0
df_pairs = pd.concat([wins, losses], ignore_index=True)
df_pairs


,Season,team1,team2,outcome
0,2003,16,16,1
1,2003,1,16,1
2,2003,10,7,1
3,2003,11,6,1
4,2003,8,9,1
...,...,...,...,...
2893,2025,2,1,0
2894,2025,2,1,0
2895,2025,1,1,0
2896,2025,1,1,0


In [19]:
len(df_pairs)==2*len(df)

True

In [20]:
wins = df[["Season", "WSeedNum", "LSeedNum", "WTeamName", "LTeamName"]].rename(
    columns={"WSeedNum": "team1", "LSeedNum": "team2", "WTeamName": "team1_name", "LTeamName": "team2_name"}
)
wins["outcome"] = 1

losses = df[["Season", "WSeedNum", "LSeedNum", "WTeamName", "LTeamName"]].rename(
    columns={"LSeedNum": "team1", "WSeedNum": "team2", "LTeamName": "team1_name", "WTeamName": "team2_name"}
)
losses["outcome"] = 0

df_pairs1 = pd.concat([wins, losses], ignore_index=True)
df_pairs1

,Season,team1,team2,team1_name,team2_name,outcome
0,2003,16,16,UNC Asheville,TX Southern,1
1,2003,1,16,Arizona,Vermont,1
2,2003,10,7,Arizona St,Memphis,1
3,2003,11,6,C Michigan,Creighton,1
4,2003,8,9,California,NC State,1
...,...,...,...,...,...,...
2893,2025,2,1,Michigan St,Auburn,0
2894,2025,2,1,Tennessee,Houston,0
2895,2025,1,1,Auburn,Florida,0
2896,2025,1,1,Duke,Houston,0


In [21]:
df_pairs1["winner"] = np.where(df_pairs1["outcome"]== 1, df_pairs1["team1_name"], df_pairs1["team2_name"])

df_pairs1

,Season,team1,team2,team1_name,team2_name,outcome,winner
0,2003,16,16,UNC Asheville,TX Southern,1,UNC Asheville
1,2003,1,16,Arizona,Vermont,1,Arizona
2,2003,10,7,Arizona St,Memphis,1,Arizona St
3,2003,11,6,C Michigan,Creighton,1,C Michigan
4,2003,8,9,California,NC State,1,California
...,...,...,...,...,...,...,...
2893,2025,2,1,Michigan St,Auburn,0,Auburn
2894,2025,2,1,Tennessee,Houston,0,Houston
2895,2025,1,1,Auburn,Florida,0,Florida
2896,2025,1,1,Duke,Houston,0,Houston


In [22]:


def winProbability(teamSeed, opponentSeed):
    if(teamSeed < 1 or teamSeed > 16 or type(teamSeed)!=int):
        print('Please enter valid seed for teamSeed')
        return -1
    if(opponentSeed < 1 or opponentSeed > 16 or type(opponentSeed)!=int):
        print('Please enter valid seed for opponentSeed')
        return -1
    seedSum = opponentSeed + teamSeed
    winProb = opponentSeed / seedSum
    return winProb

def first_round():
    regions = np.empty((4, 8))
    for i in range(4):
        results = np.empty(8)
        for j in range(8):
            if winProbability(j + 1, 16 - j) > random.uniform(0, 1):
                results[j] = int(j + 1)
            else:
                results[j] = int(16 - j)
        regions[i] = results
    return regions

def second_round(first_results):
    regions = np.empty((4, 4))
    for i in range(4):
        second_results = np.empty(4)
        for j in range(4):
            if winProbability(int(first_results[i][j]), int(first_results[i][7 - j])) > random.uniform(0, 1):
                second_results[j] = first_results[i][j]
            else:
                second_results[j] = first_results[i][7 - j]
        regions[i] = second_results
    return regions

def sweet_16(second_results):
    regions = np.empty((4, 2))
    for i in range(4):
        sweet_results = np.empty(2)
        for j in range(2):
            if winProbability(int(second_results[i][j]), int(second_results[i][3 - j])) > random.uniform(0, 1):
                sweet_results[j] = second_results[i][j]
            else:
                sweet_results[j] = second_results[i][3 - j]
        regions[i] = sweet_results
    return regions

def elite_8(sweet_results):
    regions = np.empty(4)
    for i in range(4):
        if winProbability(int(sweet_results[i][0]), int(sweet_results[i][1])) > random.uniform(0, 1):
            regions[i] = sweet_results[i][0]
        else:
            regions[i] = sweet_results[i][1]
    return regions

def final_four(elite_results):
    finals = np.empty(2)
    if winProbability(int(elite_results[0]), int(elite_results[1])) > random.uniform(0, 1):
        finals[0] = elite_results[0]
    else:
        finals[0] = elite_results[1]
    if winProbability(int(elite_results[2]), int(elite_results[3])) > random.uniform(0, 1):
        finals[1] = elite_results[2]
    else:
        finals[1] = elite_results[3]
    return finals

def championship(finals_results):
    if winProbability(int(finals_results[0]), int(finals_results[1])) > random.uniform(0, 1):
        return finals_results[0]
    else:
        return finals_results[1]

def simulate_bracket():
    first = first_round()
    second = second_round(first)
    sweet = sweet_16(second)
    elite = elite_8(sweet)
    finals = final_four(elite)
    champ = championship(finals)
    return champ

In [23]:
def simulate_bracket():
    first = first_round()
    second = second_round(first)
    sweet = sweet_16(second)
    elite = elite_8(sweet)
    finals = final_four(elite)
    champ = championship(finals)
    return champ

In [24]:
simulate_bracket()

np.float64(1.0)

In [25]:
results = [simulate_bracket() for _ in range(10000)]
pd.Series(results).value_counts(normalize=True).sort_index()

1.0     0.7452
2.0     0.1685
3.0     0.0467
4.0     0.0202
5.0     0.0083
6.0     0.0042
7.0     0.0030
8.0     0.0015
9.0     0.0011
10.0    0.0005
11.0    0.0004
12.0    0.0001
13.0    0.0001
14.0    0.0001
16.0    0.0001
Name: proportion, dtype: float64

In [26]:
champions = df.loc[df.groupby("Season")["DayNum"].idxmax(), ["Season", "WSeedNum"]]
champ_dist = champions["WSeedNum"].value_counts(normalize=True).sort_index()
champ_dist

WSeedNum
1    0.681818
2    0.090909
3    0.136364
4    0.045455
7    0.045455
Name: proportion, dtype: float64

In [27]:
comparison = pd.DataFrame({"simulated": pd.Series(results).value_counts(normalize=True), "actual": champ_dist}).fillna(0).sort_index()
comparison

,simulated,actual
1.0,0.7452,0.681818
2.0,0.1685,0.090909
3.0,0.0467,0.136364
4.0,0.0202,0.045455
5.0,0.0083,0.000000
6.0,0.0042,0.000000
7.0,0.0030,0.045455
8.0,0.0015,0.000000
9.0,0.0011,0.000000
10.0,0.0005,0.000000


In [28]:
comparison = pd.DataFrame({
    "simulated": pd.Series(results).value_counts(normalize=True),
    "actual": champ_dist,
}).fillna(0).sort_index()
comparison

,simulated,actual
1.0,0.7452,0.681818
2.0,0.1685,0.090909
3.0,0.0467,0.136364
4.0,0.0202,0.045455
5.0,0.0083,0.000000
6.0,0.0042,0.000000
7.0,0.0030,0.045455
8.0,0.0015,0.000000
9.0,0.0011,0.000000
10.0,0.0005,0.000000


In [29]:
comparison [["sim%","actual%"]] = (comparison[["simulated","actual"]]*100).astype(int)
comparison

,simulated,actual,sim%,actual%
1.0,0.7452,0.681818,74,68
2.0,0.1685,0.090909,16,9
3.0,0.0467,0.136364,4,13
4.0,0.0202,0.045455,2,4
5.0,0.0083,0.000000,0,0
6.0,0.0042,0.000000,0,0
7.0,0.0030,0.045455,0,4
8.0,0.0015,0.000000,0,0
9.0,0.0011,0.000000,0,0
10.0,0.0005,0.000000,0,0


In [ ]:
2